In [ ]:
from langchain_text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains.question_answering import load_qa_chain

In [ ]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = "sk-"

In [ ]:
#Load dos modelos

embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo", max_tokens=200)

In [ ]:
# Carregar o pdf
pdf_link = "https://www.pnas.org/content/119/7/2148/tables/t2.xml"

loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [ ]:
# Separar em chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [ ]:
# Salvar no vector db - Chroma

db = Chroma.from_documents(chunks, embedding=embeddings_model, persist_directory="text_index")
db.persist()

In [ ]:
# Carregar DB
vectordb = Chroma(persist_directory="text_index", embedding_function=embeddings_model)

# Load Retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Construção da cadeia de prompt para chamada do LLM
chain = load_qa_chain(llm, chain_type="stuff")

In [ ]:
def ask(question):
    context = retriever.get_relevant_documents(question)
    answer = (chain({"input_documents": context, "question": question}, return_only_outputs=True))['output_text']
    return answer, context

In [ ]:
user_question = input("User: ")
answer, context = ask(user_question)
print(answer)